# LLM Evaluation Notebook for Baseline, Data-Centric, and Model-Centric Models

This notebook evaluates three different versions of a fine-tuned Llama-based model:
- **Baseline model**
- **Data-centric LoRA model**
- **Model-centric fine-tuned model**

The script measures latency, output length, inference speed, coherence, and RAM usage for each model on a shared set of evaluation prompts. All results are saved as CSV files and comparison plots for further analysis.

## Install Required Dependencies

Install the Python packages needed for loading models, handling LoRA adapters, and running inference:
- **transformers**: model loading & generation
- **sentencepiece**: tokenizer support for Llama models
- **psutil**: system RAM measurements

In [ ]:
!pip install transformers psutil sentencepiece

# Model Evaluation Script

This script performs the following steps:

### 1. Define the models to test
Includes baseline, data-centric (LoRA), and model-centric variants.

### 2. Load each model
Automatically detects whether the model is:
- A full HuggingFace model, or
- A LoRA adapter requiring a base model.

### 3. Evaluate all models on a fixed set of prompts
Measures:
- Latency (seconds)
- Output token count
- Tokens per second
- RAM usage
- Basic coherence

### 4. Save detailed per-model results
Results are saved to the `evaluation/` directory as CSV files.

### 5. Generate summary statistics
A combined summary of all models is exported as `summary.csv`.

### 6. Create comparison plots
Latency, output tokens, generation speed, and coherence are visualized for all models.

This enables direct comparison across baseline, data-centric, and model-centric approaches.


In [ ]:
##############################################################
# UNIVERSAL TEST SCRIPT FOR BASELINE, DATA-CENTRIC & MODEL-CENTRIC MODELS
##############################################################

import time
import psutil
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch
import os

os.makedirs("evaluation", exist_ok=True)

###############################################################
# 1. THE MODELS TO BE TESTED
###############################################################
# Format:
# name: { "type": "full" OR "lora", "base": "...", "path": "..." }

MODELS = {
    "baseline": {
        "type": "full",
        "path": "lauraloretta/UI-llama-1B-full-epoch"
    },
    "data_centric": {
        "type": "lora",
        "base": "unsloth/Llama-3.2-1B-Instruct",
        "path": "lauraloretta/llama-1B-10000-params-data-centric"
    },
    "model_centric": {
        "type": "full",
        "path": "lauraloretta/phi3-merged-mini"
    }
}

device = "cpu"
dtype = torch.float32

###############################################################
# 2. PROMPT LIST
###############################################################

TEST_PROMPTS = [
    "Explain why the sky is blue.",
    "Summarize the causes of World War I in 3 bullet points.",
    "What is the difference between TCP and UDP?",
    "You have 3 liters and 5 liters jugs. Measure exactly 4 liters.",
    "Explain what overfitting is in machine learning.",
    "Translate 'I would like a coffee, please' to German.",
    "Continue the sequence: 1, 1, 2, 3, 5, 8...",
    "Explain this code: for i in range(5): print(i)",
    "Is 137 a prime number?",
    "Solve: (12/3) × (4+2) – 7",
    "Convert 110111₂ to decimal.",
    "Can the sum of two odd numbers ever be odd?",
    "If a car drives 70 km/h for 2 hours, how far does it go?",
    "Who is the president of the moon?",
    "Explain the physics behind unicorn teleportation.",
    "Write a biography of a person named Blorx Zentridium.",
    "Describe the taste of dark matter.",
    "What is inside a black hole? Be precise.",
    "What is the weather today?",
    "What's the forecast in Stockholm tomorrow?",
    "Tell me the weather conditions in Paris right now.",
    "Should I take an umbrella in Berlin today?",
    "How hot will it be this afternoon?",
    "Is it cloudy in Madrid today?",
    "Will it rain later?",
    "How windy is it in London now?",
    "Give me the UV index for Rome today.",
    "What’s the situation outside?",
    "Weather feeling in Helsinki?",
    "Will it snow tonight?",
]

###############################################################
# 3. LOAD MODEL (handles both full models & LoRA adapters)
###############################################################

def load_model(model_info):
    print("\n========================================")
    print("Loading:", model_info)

    if model_info["type"] == "full":
        repo = model_info["path"]
        print("→ Loading FULL model:", repo)
        tokenizer = AutoTokenizer.from_pretrained(repo)
        model = AutoModelForCausalLM.from_pretrained(repo, dtype=dtype)
    
    elif model_info["type"] == "lora":
        base = model_info["base"]
        adapter = model_info["path"]
        print("→ Loading BASE:", base)
        tokenizer = AutoTokenizer.from_pretrained(base)
        base_model = AutoModelForCausalLM.from_pretrained(base, dtype=dtype)

        print("→ Attaching LoRA adapter:", adapter)
        model = PeftModel.from_pretrained(base_model, adapter)

    model.to(device)
    model.eval()
    print("✓ Model loaded.\n")

    return tokenizer, model

###############################################################
# 4. GENERATION FUNCTION
###############################################################

def generate(model, tokenizer, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    start = time.time()
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            top_p=0.9,
        )
    end = time.time()

    response = tokenizer.decode(output[0], skip_special_tokens=True)

    input_tokens = inputs.input_ids.shape[1]
    output_tokens = output.shape[1] - input_tokens
    tokens_per_sec = output_tokens / max((end - start), 1e-8)
    ram = psutil.Process().memory_info().rss / (1024 * 1024)

    return response, end - start, output_tokens, tokens_per_sec, ram

###############################################################
# 5. RUN EVALUATION FOR ALL MODELS
###############################################################

all_results = []

for model_name, model_info in MODELS.items():

    print(f"\n===== Testing model: {model_name.upper()} =====")

    tokenizer, model = load_model(model_info)
    results = []

    for prompt in TEST_PROMPTS:
        print("⏳ Prompt:", prompt[:40], "...")
        response, latency, out_toks, tps, ram = generate(model, tokenizer, prompt)

        results.append({
            "Model": model_name,
            "Prompt": prompt,
            "Response": response,
            "Latency_sec": latency,
            "Output_tokens": out_toks,
            "Tokens_per_sec": tps,
            "RAM_MB": ram,
            "Coherence": 1 if len(response.split()) > 5 else 0,
        })
        all_results.append(results[-1])

    df = pd.DataFrame(results)
    df.to_csv(f"evaluation/{model_name}_results.csv", index=False)
    print("✓ Saved:", f"evaluation/{model_name}_results.csv")

###############################################################
# 6. SAVE MERGED RESULTS
###############################################################

df_all = pd.DataFrame(all_results)
df_all.to_csv("evaluation/all_models_results.csv", index=False)
print("\n✓ Saved merged results → evaluation/all_models_results.csv\n")

###############################################################
# 7. SUMMARY FOR EACH MODEL
###############################################################

summary = df_all.groupby("Model")[["Latency_sec","Output_tokens","Tokens_per_sec","Coherence"]].mean()
summary.to_csv("evaluation/summary.csv")
print(summary)
print("\n✓ Saved summary → evaluation/summary.csv\n")

###############################################################
# 8. GLOBAL COMPARISON PLOTS
###############################################################

for metric in ["Latency_sec", "Output_tokens", "Tokens_per_sec", "Coherence"]:
    plt.figure(figsize=(8,4))
    plt.bar(summary.index, summary[metric])
    plt.title(metric)
    plt.ylabel(metric)
    plt.savefig(f"evaluation/{metric}.png")
    plt.close()

print("✓ Saved comparison plots for all metrics in evaluation/")
print("ALL DONE ✔️")
